In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np

data = pd.read_csv("weather_data.csv")

print(data.columns)# Convert DATE to actual datetime
data['DATE'] = pd.to_datetime(data['DATE'])

# Sort chronologically
data = data.sort_values('DATE')

# Remove columns we don't need
data = data.drop(columns=['STATION', 'NAME', 'SNWD'])

print(data.head())



Index(['STATION', 'NAME', 'DATE', 'PRCP', 'SNWD', 'TMAX', 'TMIN'], dtype='str')
        DATE  PRCP  TMAX  TMIN
0 1985-11-01  0.00    45    37
1 1985-11-02  0.00    47    38
2 1985-11-03  0.00    44    32
3 1985-11-04  0.11    40    28
4 1985-11-05  0.07    61    38


In [138]:
data['TMAX'] = (data['TMAX'] - 32) * 5/9
data['TMIN'] = (data['TMIN'] - 32) * 5/9
print(data.head())
print(data.describe())

        DATE  PRCP       TMAX      TMIN
0 1985-11-01  0.00   7.222222  2.777778
1 1985-11-02  0.00   8.333333  3.333333
2 1985-11-03  0.00   6.666667  0.000000
3 1985-11-04  0.11   4.444444 -2.222222
4 1985-11-05  0.07  16.111111  3.333333
                             DATE          PRCP          TMAX          TMIN
count                       14791  14791.000000  14791.000000  14791.000000
mean   2006-03-24 04:37:16.278818      0.070909     15.624178      7.667613
min           1985-11-01 00:00:00      0.000000    -11.111111    -15.000000
25%           1995-12-16 12:00:00      0.000000      8.888889      2.777778
50%           2006-04-01 00:00:00      0.000000     15.555556      7.777778
75%           2016-06-15 12:00:00      0.070000     22.222222     12.777778
max           2026-07-31 00:00:00      2.420000     41.111111     25.000000
std                           NaN      0.158162      8.812848      6.485582


In [139]:
features = data[['PRCP', 'TMAX', 'TMIN']].values
X = []
y = []

sequence_length = 14

for i in range(len(features) - sequence_length):

    X.append(
        features[i:i + sequence_length]
    )

    y.append(
        features[i + sequence_length, 1]
    )
print(X[0])
print(y[0])

X = np.array(X)
y = np.array(y)

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

y = y.reshape(-1, 1)

print(X.shape)
print(y.shape)
train_size = int(0.70 * len(X))

val_size = int(0.15 * len(X))

X_train = X[:train_size]
y_train = y[:train_size]

X_val = X[train_size:train_size + val_size]
y_val = y[train_size:train_size + val_size]

X_test = X[train_size + val_size:]
y_test = y[train_size + val_size:]

[[ 0.          7.22222222  2.77777778]
 [ 0.          8.33333333  3.33333333]
 [ 0.          6.66666667  0.        ]
 [ 0.11        4.44444444 -2.22222222]
 [ 0.07       16.11111111  3.33333333]
 [ 0.03       11.11111111  4.44444444]
 [ 0.06       10.          3.88888889]
 [ 0.04       10.          7.77777778]
 [ 0.3        16.11111111  9.44444444]
 [ 0.05       12.77777778  5.        ]
 [ 0.          6.66666667  0.55555556]
 [ 0.          4.44444444  1.11111111]
 [ 0.          3.33333333 -1.11111111]
 [ 0.          3.88888889 -2.22222222]]
0.5555555555555556
torch.Size([14777, 14, 3])
torch.Size([14777, 1])


In [140]:
print("Training:")
print(X_train.shape)
print(y_train.shape)

print("Validation:")
print(X_val.shape)
print(y_val.shape)

print("Testing:")
print(X_test.shape)
print(y_test.shape)

Training:
torch.Size([10343, 14, 3])
torch.Size([10343, 1])
Validation:
torch.Size([2216, 14, 3])
torch.Size([2216, 1])
Testing:
torch.Size([2218, 14, 3])
torch.Size([2218, 1])


In [141]:
X_mean = X_train.mean(dim=(0, 1), keepdim=True)
X_std = X_train.std(dim=(0, 1), keepdim=True)

X_train = (X_train - X_mean) / X_std
X_val = (X_val - X_mean) / X_std
X_test = (X_test - X_mean) / X_std

y_mean = y_train.mean()
y_std = y_train.std()

y_train = (y_train - y_mean) / y_std
y_val = (y_val - y_mean) / y_std
y_test = (y_test - y_mean) / y_std

In [142]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

trainloader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

valloader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

testloader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)


In [143]:
import torch
import torch.nn as nn

class WeatherModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(42, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

        self.relu = nn.ReLU()

    def forward(self, x):
        # x shape: [batch_size, 14, 3]

        x = x.reshape(x.size(0), -1)

        # now x shape: [batch_size, 42]

        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))

        x = self.fc3(x)

        return x
model = WeatherModel()   
print(model)

WeatherModel(
  (fc1): Linear(in_features=42, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=1, bias=True)
  (relu): ReLU()
)


In [144]:
model = WeatherModel()

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 40

for epoch in range(epochs):

    # ----------------------
    # Training
    # ----------------------
    model.train()

    train_loss = 0.0

    for X_batch, y_batch in trainloader:

        predictions = model(X_batch)

        loss = criterion(predictions, y_batch)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss = train_loss / len(trainloader)

    # ----------------------
    # Validation
    # ----------------------
    model.eval()

    val_loss = 0.0

    with torch.no_grad():

        for X_batch, y_batch in valloader:

            predictions = model(X_batch)

            loss = criterion(predictions, y_batch)

            val_loss += loss.item()

    val_loss = val_loss / len(valloader)

    print(
        f"Epoch {epoch+1}/{epochs} "
        f"Train Loss: {train_loss:.4f} "
        f"Validation Loss: {val_loss:.4f}"
    )

Epoch 1/40 Train Loss: 0.1854 Validation Loss: 0.1225
Epoch 2/40 Train Loss: 0.1044 Validation Loss: 0.1140
Epoch 3/40 Train Loss: 0.0978 Validation Loss: 0.1108
Epoch 4/40 Train Loss: 0.0950 Validation Loss: 0.1079
Epoch 5/40 Train Loss: 0.0932 Validation Loss: 0.1100
Epoch 6/40 Train Loss: 0.0918 Validation Loss: 0.1093
Epoch 7/40 Train Loss: 0.0913 Validation Loss: 0.1094
Epoch 8/40 Train Loss: 0.0899 Validation Loss: 0.1089
Epoch 9/40 Train Loss: 0.0889 Validation Loss: 0.1088
Epoch 10/40 Train Loss: 0.0886 Validation Loss: 0.1081
Epoch 11/40 Train Loss: 0.0884 Validation Loss: 0.1087
Epoch 12/40 Train Loss: 0.0866 Validation Loss: 0.1098
Epoch 13/40 Train Loss: 0.0865 Validation Loss: 0.1087
Epoch 14/40 Train Loss: 0.0863 Validation Loss: 0.1090
Epoch 15/40 Train Loss: 0.0855 Validation Loss: 0.1098
Epoch 16/40 Train Loss: 0.0854 Validation Loss: 0.1108
Epoch 17/40 Train Loss: 0.0842 Validation Loss: 0.1077
Epoch 18/40 Train Loss: 0.0823 Validation Loss: 0.1081
Epoch 19/40 Train L

In [145]:
model.eval()

test_loss = 0.0
predictions_list = []
actual_list = []

with torch.no_grad():

    for X_batch, y_batch in testloader:

        predictions = model(X_batch)

        loss = criterion(predictions, y_batch)
        test_loss += loss.item()

        predictions_list.append(predictions)
        actual_list.append(y_batch)

test_loss = test_loss / len(testloader)

print("Test Loss:", test_loss)

predictions = torch.cat(predictions_list)
actual = torch.cat(actual_list)
predictions_real = predictions * y_std + y_mean
actual_real = actual * y_std + y_mean

mae = torch.mean(torch.abs(predictions_real - actual_real))

rmse = torch.sqrt(
    torch.mean((predictions_real - actual_real) ** 2)
)

print("MAE:", mae.item())
print("RMSE:", rmse.item())


Test Loss: 0.11848492228559085
MAE: 2.3472447395324707
RMSE: 3.00473690032959
